# 📹 Notebook 05: CCTV Accident Detection & Edge Computer Vision

**RoadRisk AI Portfolio Project — Module H (Phase 8)**

This notebook demonstrates:
1. **Dataset Manifest & Integrity**: Scanning 990 CCTV frames across `data/train`, `data/val`, and `data/test` splits.
2. **Data Augmentation & Preprocessing**: Torchvision transform pipelines for training vs evaluation.
3. **MobileNetV2 Transfer Learning**: Lightweight CNN architecture (~9MB footprint) optimized for edge deployment.
4. **Model Evaluation**: Test set confusion matrix, precision, recall, macro-F1, and ROC-AUC.
5. **Grad-CAM Visual Attention**: Explaining neural network predictions via class activation maps.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import torch
import torchvision
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.cctv_preprocessing import scan_cctv_manifest, CLASS_NAMES
from src.cctv_inference import load_cctv_model, predict_cctv_frame, generate_cctv_gradcam, CCTV_METRICS_PATH
from src.config import CCTV_TEST_DIR

print(f"PyTorch: {torch.__version__} | Torchvision: {torchvision.__version__}")

## 1. Scan CCTV Dataset Manifest

In [ ]:
df_manifest = scan_cctv_manifest(verify_images=False)
print(f"Total Images indexed: {len(df_manifest)}")
print(df_manifest.groupby(["split", "class_name"]).size())
df_manifest.head()

## 2. Load Evaluation Metrics & Confusion Matrix

In [ ]:
with open(CCTV_METRICS_PATH, "r", encoding="utf-8") as f:
    metrics = json.load(f)

tm = metrics["test_metrics"]
print(f"Test Accuracy: {tm['accuracy']*100:.2f}%")
print(f"Macro-F1 Score: {tm['macro_f1']:.4f}")
print(f"ROC-AUC Score: {tm['roc_auc']:.4f}")
print("Confusion Matrix (Actual rows x Predicted cols):")
print(np.array(tm["confusion_matrix"]))

# Display training loss curves
h_df = pd.DataFrame(metrics["training_history"])
plt.figure(figsize=(8, 4))
plt.plot(h_df["epoch"], h_df["train_loss"], label="Train Loss", marker="o")
plt.plot(h_df["epoch"], h_df["val_loss"], label="Val Loss", marker="s")
plt.title("Fine-Tuning Loss Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Single-Frame Inference & Grad-CAM Visual Explainability

In [ ]:
model, device = load_cctv_model()

# Select a sample test image
sample_path = list((CCTV_TEST_DIR / "Accident").glob("*.*"))[0]
orig_img = Image.open(sample_path)

# Run inference
res = predict_cctv_frame(sample_path, model=model, device=device)
print("Prediction Result:", res)

# Generate Grad-CAM
overlay, cam = generate_cctv_gradcam(sample_path, model=model, device=device)

# Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(orig_img)
axes[0].set_title(f"Original Feed ({res['prediction']})")
axes[0].axis("off")

axes[1].imshow(overlay)
axes[1].set_title(f"Grad-CAM Heatmap ({res['confidence']*100:.1f}% Conf)")
axes[1].axis("off")

plt.tight_layout()
plt.show()